In [ ]:
#843k
"""
=================================================================
SALES FORECASTING  –  GIẢI PHÁP TỐI ƯU v4.7 (THE SNIPER RIFLE)
-----------------------------------------------------------------
Phát hiện chính:
  • sample_submission chỉ là template (dummy values).
  • Cần Time-Series CV thật: cut 2019/2020/2021 → val 2020/2021/2022.
  • Structural break 2018→2019 (-38%), model cần lag features ổn định.

Kiến trúc hai tầng:
  MODEL A  –  Direct HGB (lag365/730 + Fourier)  × 5 seeds
  MODEL B  –  Monthly decompose + Daily share
              • Predict monthly total (ít nhiễu hơn daily)
              • Distribute qua predicted daily_share pattern

  Final = 0.35 × A  +  0.65 × B   (Ép mạnh vào sự ổn định của Model B)

Iterative 2-step cho test:
  Step 1: Predict 2023  (lag365 = train 2022, coverage 100 %)
  Step 2: Predict 2024  (lag365 = pred 2023, iterative)

🔥 BƯỚC ĐỘT PHÁ v4.7: 
  - Khôi phục bộ giảm xóc 1.5x đã tạo nên kỷ lục 846k.
  - Tối ưu hóa hàm Loss Model A: 60% MAE + 40% RMSE (Bắn thẳng vào hàm mục tiêu LB).
=================================================================
"""

import pandas as pd
import numpy as np
import os
import warnings
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# ──────────────────────────────────────────
# 0.  CẤU HÌNH
# ──────────────────────────────────────────
RAW_PATH        = "../data/raw/"
SUBMISSION_PATH = "../submission/"
SEEDS           = [42, 123, 2024, 999, 7]

# 🌟 Tinh chỉnh trọng số: Ép mạnh vào sự ổn định của xu hướng tháng
WEIGHT_A        = 0.35   
WEIGHT_B        = 0.65   

os.makedirs(SUBMISSION_PATH, exist_ok=True)

# ──────────────────────────────────────────
# 1.  TẢI DỮ LIỆU
# ──────────────────────────────────────────
print("=" * 60)
print("1. TẢI DỮ LIỆU")
print("=" * 60)

train_raw  = pd.read_csv(f"{RAW_PATH}sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sample_sub = pd.read_csv(f"{RAW_PATH}sample_submission.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

train_raw["cogs_ratio"] = train_raw["COGS"] / train_raw["Revenue"]
train_raw["dom"]        = train_raw["Date"].dt.day
train_raw["dow"]        = train_raw["Date"].dt.dayofweek
train_raw["month_col"]  = train_raw["Date"].dt.month
train_raw["year"]       = train_raw["Date"].dt.year
train_raw["ym"]         = train_raw["Date"].dt.to_period("M")

# Để nguyên toàn bộ lịch sử 2012-2022 để AI tra cứu Lag 24 tháng
sr        = dict(zip(train_raw["Date"], train_raw["Revenue"]))
sr_cogs   = dict(zip(train_raw["Date"], train_raw["cogs_ratio"]))

print(f"Train  : {len(train_raw)} ngày | "
      f"{train_raw['Date'].min().date()} → {train_raw['Date'].max().date()}")
print(f"Test   : {len(sample_sub)} ngày | "
      f"{sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()}")
print(f"Lưu ý  : sample_submission là template – values không phải ground truth")

# ──────────────────────────────────────────
# 2A.  FEATURES CHO MODEL A  (Direct HGB)
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("2. FEATURE ENGINEERING")
print("=" * 60)

def make_direct_features(dates_list: list, sd: dict) -> pd.DataFrame:
    rows = []
    for dt in dates_list:
        r = {}
        r["year"]    = dt.year
        r["month"]   = dt.month
        r["day"]     = dt.day
        r["dow"]     = dt.dayofweek
        r["doy"]     = dt.dayofyear
        r["qtr"]     = dt.quarter
        r["dim"]     = dt.days_in_month
        r["dte"]     = r["dim"] - r["day"]
        r["istart"]  = int(r["day"] <= 2)
        r["iend"]    = int(r["dte"] <= 1)
        r["yrc"]     = r["year"] - 2017

        doy = r["doy"]; dow = r["dow"]; day = r["day"]
        for k in range(1, 7):
            r[f"sy{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
            r[f"cy{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
        for k in range(1, 4):
            r[f"sw{k}"] = np.sin(2 * np.pi * k * dow / 7)
            r[f"cw{k}"] = np.cos(2 * np.pi * k * dow / 7)
            r[f"sm{k}"] = np.sin(2 * np.pi * k * day / 31)
            r[f"cm{k}"] = np.cos(2 * np.pi * k * day / 31)

        for off in range(-4, 5):
            r[f"L365o{off}"] = sd.get(dt - pd.Timedelta(days=365 - off), np.nan)
            r[f"L730o{off}"] = sd.get(dt - pd.Timedelta(days=730 - off), np.nan)

        for w in [3, 7, 14, 30, 60, 90]:
            v = [sd.get(dt - pd.Timedelta(days=365 + i), np.nan) for i in range(-w, w + 1)]
            v = [x for x in v if not np.isnan(x)]
            r[f"R365w{w}"]  = np.mean(v)   if v else np.nan
            r[f"Rm365w{w}"] = np.median(v) if v else np.nan

        for w in [3, 7, 14, 30, 60, 90]:
            v = [sd.get(dt - pd.Timedelta(days=730 + i), np.nan) for i in range(-w, w + 1)]
            v = [x for x in v if not np.isnan(x)]
            r[f"R730w{w}"]  = np.mean(v)   if v else np.nan
            r[f"Rm730w{w}"] = np.median(v) if v else np.nan

        rows.append(r)
    return pd.DataFrame(rows)


# ──────────────────────────────────────────
# 2B.  FEATURES CHO MODEL B  (Monthly decompose)
# ──────────────────────────────────────────

month_totals = train_raw.groupby("ym")["Revenue"].sum().to_dict()
train_raw["daily_share"] = train_raw.apply(lambda row: row["Revenue"] / month_totals.get(row["ym"], 1e-9), axis=1)

SHARE_FEAT = ["dom", "dow", "month_col"]

def make_monthly_features(ym_list: list, monthly_dict: dict) -> pd.DataFrame:
    rows = []
    for ym in ym_list:
        r = {}
        r["year"]  = ym.year
        r["month"] = ym.month
        r["yrc"]   = ym.year - 2017
        for k in range(1, 5):
            r[f"sm{k}"] = np.sin(2 * np.pi * k * ym.month / 12)
            r[f"cm{k}"] = np.cos(2 * np.pi * k * ym.month / 12)
        for lag in [1, 2, 3, 6, 12, 13, 14, 24, 25]:
            r[f"Ml{lag}"] = monthly_dict.get(ym - lag, np.nan)
        for w in [3, 6, 12]:
            v = [monthly_dict.get(ym - l, np.nan) for l in range(1, w + 1)]
            v = [x for x in v if not np.isnan(x)]
            r[f"Mr{w}"] = np.mean(v) if v else np.nan
        rows.append(r)
    return pd.DataFrame(rows)


# ──────────────────────────────────────────
# 3.  HÀM DỰ BÁO TỔNG HỢP
# ──────────────────────────────────────────

def predict_one_year(
    dates: pd.Series,
    sr_dict: dict,
    sr_ratio_dict: dict,
    train_data: pd.DataFrame,
    seeds: list = SEEDS,
) -> tuple:
    # ── Model A: Direct HGB ──────────────────────────────────
    tf = make_direct_features(train_data["Date"].tolist(), sr_dict)
    tf["Revenue"] = train_data["Revenue"].values
    trv = tf.dropna(subset=["R730w3"]).copy()
    
    trv = trv[trv["year"] >= 2021] 
    
    FEAT_A = [c for c in trv.columns if c != "Revenue"]
    vf = make_direct_features(dates.tolist(), sr_dict)

    pA_list = []
    for s in seeds:
        m_mae = HistGradientBoostingRegressor(loss="absolute_error", max_iter=600, learning_rate=0.02, max_depth=6, min_samples_leaf=10, l2_regularization=0.1, random_state=s)
        m_rmse = HistGradientBoostingRegressor(loss="squared_error", max_iter=600, learning_rate=0.02, max_depth=6, min_samples_leaf=10, l2_regularization=0.1, random_state=s)
        
        m_mae.fit(trv[FEAT_A], trv["Revenue"])
        m_rmse.fit(trv[FEAT_A], trv["Revenue"])
        
        # 🌟 VŨ KHÍ MỚI: Trọng số 60% MAE + 40% RMSE (Đánh mạnh vào hàm tính điểm LB)
        blend_pred = (m_mae.predict(vf[FEAT_A]) * 0.6) + (m_rmse.predict(vf[FEAT_A]) * 0.4)
        pA_list.append(blend_pred)
        print(f"    A seed={s} ✓")
    pA = np.mean(pA_list, axis=0)

    # ── Model B: Monthly decompose ───────────────────────────
    month_totals_tr = train_data.groupby("ym")["Revenue"].sum().to_dict()
    train_data_b = train_data.copy()
    train_data_b["daily_share_b"] = train_data_b.apply(lambda r: r["Revenue"] / month_totals_tr.get(r["ym"], 1e-9), axis=1)
    
    train_data_b = train_data_b[train_data_b["year"] >= 2021]

    hgb_share = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, max_depth=4, min_samples_leaf=20, random_state=42)
    hgb_share.fit(train_data_b[SHARE_FEAT], train_data_b["daily_share_b"])

    monthly_dict = {ym: grp["Revenue"].sum() for ym, grp in train_data.groupby("ym")}
    tr_yms = sorted(monthly_dict.keys())
    mf_tr  = make_monthly_features(tr_yms, monthly_dict)
    mf_tr["Rev"] = [monthly_dict[ym] for ym in tr_yms]
    mf_tr_v = mf_tr.dropna(subset=["Ml24"]).copy()
    
    mf_tr_v = mf_tr_v[mf_tr_v["year"] >= 2021]
    
    MFEAT = [c for c in mf_tr_v.columns if c != "Rev"]
    val_yms_unique = sorted(dates.dt.to_period("M").unique().tolist())
    mf_val = make_monthly_features(val_yms_unique, monthly_dict)

    hgb_m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.03, max_depth=5, min_samples_leaf=5, random_state=42)
    hgb_m.fit(mf_tr_v[MFEAT], mf_tr_v["Rev"])
    pred_monthly_map = dict(zip([ym.month + ym.year * 100 for ym in val_yms_unique], hgb_m.predict(mf_val[MFEAT])))

    val_df = pd.DataFrame({
        "Date": dates.values,
        "dom":  dates.dt.day.values,
        "dow":  dates.dt.dayofweek.values,
        "month_col": dates.dt.month.values,
        "ym":   dates.dt.to_period("M").values,
    })
    val_df["pred_monthly"] = (val_df["Date"].dt.year * 100 + val_df["Date"].dt.month).map(pred_monthly_map)
    val_df["pred_share"] = hgb_share.predict(val_df[SHARE_FEAT])
    share_sum = val_df.groupby("ym")["pred_share"].transform("sum")
    val_df["pred_share_norm"] = val_df["pred_share"] / share_sum
    pB = val_df["pred_monthly"].values * val_df["pred_share_norm"].values

    # ── Ensemble ─────────────────────────────────────────────
    rev_preds = WEIGHT_A * pA + WEIGHT_B * pB

    # ── COGS ratio (Model A only) ────────────────────────────
    tf_c = make_direct_features(train_data["Date"].tolist(), sr_ratio_dict)
    tf_c["cogs_ratio"] = train_data["cogs_ratio"].values
    trv_c = tf_c.dropna(subset=["R730w3"]).copy()
    
    trv_c = trv_c[trv_c["year"] >= 2021]
    
    FEAT_C = [c for c in trv_c.columns if c != "cogs_ratio"]
    vf_c = make_direct_features(dates.tolist(), sr_ratio_dict)

    ratio_list = []
    for s in seeds[:3]:
        m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.03, max_depth=5, min_samples_leaf=10, l2_regularization=0.1, random_state=s)
        m.fit(trv_c[FEAT_C], trv_c["cogs_ratio"])
        ratio_list.append(m.predict(vf_c[FEAT_C]))
        print(f"    C seed={s} ✓")
    ratio_preds = np.mean(ratio_list, axis=0)

    return rev_preds, ratio_preds


# ──────────────────────────────────────────
# 4.  VALIDATION OFFLINE
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("3. VALIDATION OFFLINE (Skipped for Speed)")
print("=" * 60)


# ──────────────────────────────────────────
# 5.  DỰ BÁO TEST – ITERATIVE 2 BƯỚC
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("4. DỰ BÁO TEST – ITERATIVE 2 BƯỚC")
print("=" * 60)

sub23 = sample_sub[sample_sub["Date"].dt.year == 2023].copy()
sub24 = sample_sub[sample_sub["Date"].dt.year == 2024].copy()

print("\n[Bước 1] Predict 2023 (lag365 = train 2022 – coverage 100 %)")
rev23, ratio23 = predict_one_year(sub23["Date"], sr, sr_cogs, train_raw)

sr_ext      = {**sr,      **dict(zip(sub23["Date"], rev23))}
sr_cogs_ext = {**sr_cogs, **dict(zip(sub23["Date"], ratio23))}

print("\n[Bước 2] Predict 2024 (lag365 = pred_2023 – iterative)")
rev24, ratio24 = predict_one_year(sub24["Date"], sr_ext, sr_cogs_ext, train_raw)

# ──────────────────────────────────────────
# 6.  TẠO FILE SUBMISSION
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("5. TẠO FILE SUBMISSION")
print("=" * 60)

all_rev   = np.concatenate([rev23,   rev24])
all_ratio = np.concatenate([ratio23, ratio24])

sub_out = sample_sub[["Date"]].copy()

GROWTH_FACTOR = 1.38 

# 🌟 KHÔI PHỤC CHỐT CHẶN VÀNG: Clip đúng 1.5x để hấp thụ xung lực dội ngược của AI
sub_out["Revenue"] = np.clip(all_rev * GROWTH_FACTOR, 0, train_raw["Revenue"].max() * 1.5).round(2)
sub_out["COGS"]    = (sub_out["Revenue"] * np.clip(all_ratio, 0.55, 1.15)).round(2)

sub_out = sub_out.set_index("Date").loc[sample_sub["Date"]].reset_index()

out_path = f"{SUBMISSION_PATH}submission.csv"
sub_out.to_csv(out_path, index=False)

print(f"  Revenue mean : {sub_out['Revenue'].mean():>12,.0f}")
print(f"  Revenue min  : {sub_out['Revenue'].min():>12,.0f}")
print(f"  Revenue max  : {sub_out['Revenue'].max():>12,.0f}")
print(f"  COGS/Rev     : {(sub_out['COGS']/sub_out['Revenue']).mean():>12.4f}")

assert list(sub_out.columns) == ["Date", "Revenue", "COGS"]
assert len(sub_out) == len(sample_sub)
assert (sub_out["Revenue"] >= 0).all()
print(f"\n✅ Lưu: {out_path}  ({len(sub_out)} dòng)")

1. TẢI DỮ LIỆU
Train  : 3833 ngày | 2012-07-04 → 2022-12-31
Test   : 548 ngày | 2023-01-01 → 2024-07-01
Lưu ý  : sample_submission là template – values không phải ground truth

2. FEATURE ENGINEERING

3. VALIDATION OFFLINE (Skipped for Speed)

4. DỰ BÁO TEST – ITERATIVE 2 BƯỚC

[Bước 1] Predict 2023 (lag365 = train 2022 – coverage 100 %)
